In [1]:
# ============================================================
# TRANSFORM DRIVERS DATA — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql import functions as F
from pyspark.sql.types import *
import nbformat
from nbconvert import PythonExporter

In [2]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/03_silver_helpers.ipynb")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 13:48:53 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/09 13:48:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 13:48:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 13:48:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/09 13:48:53 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/09 13:48:53 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/09 13:48:53 

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [3]:
# -----------------------------------------
# 2. Receive p_batch_id (unified logic)
# -----------------------------------------

# Case 1: Papermill parameter
if "p_batch_id" in globals() and p_batch_id not in [None, ""]:
    pass

# Case 2: Databricks widget
elif "dbutils" in globals():
    p_batch_id = dbutils.widgets.get("p_batch_id")

# Case 3: Manual Jupyter execution → auto-detect latest batch
else:
    bronze_folder = f"{BRONZE_PATH}/drivers/data.parquet"
    batch_folders = [
        f.split("=")[1]
        for f in os.listdir(bronze_folder)
        if f.startswith("batch_id=")
    ]
    if not batch_folders:
        raise Exception("❌ No batch_id partitions found in Bronze drivers")
    p_batch_id = sorted(batch_folders)[-1]
    print("⚠️ Auto-selected latest batch_id:", p_batch_id)

# Final validation
if p_batch_id is None or p_batch_id == "":
    raise Exception("❌ p_batch_id not provided to Silver notebook")

print("Using p_batch_id:", p_batch_id)

⚠️ Auto-selected latest batch_id: 20260609_131505
Using p_batch_id: 20260609_131505


In [4]:
# -----------------------------------------
# 3. Define Bronze + Silver paths
# -----------------------------------------
bronze_path = f"{BRONZE_PATH}/drivers/data.parquet"
silver_path = f"{SILVER_PATH}/drivers"
os.makedirs(silver_path, exist_ok=True)

In [5]:
# -----------------------------------------
# 4. Read Bronze (ONLY this batch)
# -----------------------------------------
drivers_df = (
    spark.read.parquet(bronze_path)
    .filter(F.col("batch_id") == p_batch_id)
)

print("✔ Bronze drivers read")
drivers_df.show(5, truncate=False)

✔ Bronze drivers read


+---------+-------------------+-----------+-----------+----------------------------------------------+--------------------------+-------------------------------------------------------------------+---------------+
|driverId |name               |dateOfBirth|nationality|url                                           |ingest_timestamp          |source_file                                                        |batch_id       |
+---------+-------------------+-----------+-----------+----------------------------------------------+--------------------------+-------------------------------------------------------------------+---------------+
|abate    |{carlo, abate}     |1932-07-10 |italian    |http://en.wikipedia.org/wiki/Carlo_Mario_Abate|2026-06-09 13:15:07.060106|/Users/manoharazalki/F1-Analytics/data/landing/2025-01/drivers.json|20260609_131505|
|abecassis|{george, abecassis}|1913-03-21 |british    |http://en.wikipedia.org/wiki/George_Abecassis |2026-06-09 13:15:07.060106|/Users/manohara

In [6]:
# -----------------------------------------
# 5. Drop URL column
# -----------------------------------------
drivers_dropped_df = drivers_df.drop("url")

In [7]:
# -----------------------------------------
# 6. Standardize + rename columns
# -----------------------------------------
drivers_renamed_df = (
    drivers_dropped_df
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("dateOfBirth", "date_of_birth")
)

In [8]:
# -----------------------------------------
# 7. Concatenate givenName + familyName → driver_name
# -----------------------------------------
drivers_concatenated_df = (
    drivers_renamed_df
        .withColumn(
            "driver_name",
            F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))
        )
        .drop("name")
)

In [9]:
# -----------------------------------------
# 8. Remove duplicates
# -----------------------------------------
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])

In [10]:
# -----------------------------------------
# 9. Title Case transformation
# -----------------------------------------
drivers_final_df = (
    drivers_distinct_df
        .withColumn("driver_name", F.initcap("driver_name"))
        .withColumn("nationality", F.initcap("nationality"))
)

In [11]:
# -----------------------------------------
# 10. Write to Silver
# -----------------------------------------
write_to_silver(
    drivers_final_df,
    f"{silver_path}/data.parquet",
    merge_keys=["driver_id"]
)

print("✔ Drivers Silver written:", f"{silver_path}/data.parquet")

✔ Drivers Silver written: /Users/manoharazalki/F1-Analytics/data/silver/drivers/data.parquet


In [12]:
# -----------------------------------------
# 11. Validate
# -----------------------------------------
spark.read.parquet(f"{silver_path}/data.parquet").show(5, truncate=False)

+---------+-------------+-----------+--------------------------+-------------------------------------------------------------------+---------------+----------------+--------------------------+--------------------------+
|driver_id|date_of_birth|nationality|ingest_timestamp          |source_file                                                        |batch_id       |driver_name     |created_timestamp         |updated_timestamp         |
+---------+-------------+-----------+--------------------------+-------------------------------------------------------------------+---------------+----------------+--------------------------+--------------------------+
|Cannoc   |1933-06-21   |Canadian   |2026-06-09 13:15:07.060106|/Users/manoharazalki/F1-Analytics/data/landing/2025-01/drivers.json|20260609_131505|John Cannon     |2026-06-09 13:48:57.683215|2026-06-09 13:48:57.683215|
|Changy   |1922-02-05   |Belgian    |2026-06-09 13:15:07.060106|/Users/manoharazalki/F1-Analytics/data/landing/2025-01/d